In [ ]:
import pandas as pd
import requests
import os
import re

os.environ["HF_TOKEN"] = "SECRET"

In [ ]:
from tqdm.auto import tqdm 
from datasets import load_dataset

# Lấy lại api_url từ dataset gốc
ds = load_dataset("tmquan/vbpl-vn", split="train", token=os.environ["HF_TOKEN"])


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [3]:
import re
import pandas as pd
import numpy as np
import json


def _normalize_doc_number(doc_numbers):
    """
    [" 3131/QĐ-UB-NCVX ", "12 / TT-BTC"]
    -> "3131/QĐ-UB-NCVX, 12/TT-BTC"
    """

    if doc_numbers is None:
        return None
    if not isinstance(doc_numbers, list) :
        doc_numbers = [doc_numbers]

    cleaned = []

    for x in doc_numbers:
        if x is None:
            continue

        # xóa toàn bộ khoảng trắng
        x = re.sub(r"\s+", "", str(x))

        if x:
            cleaned.append(x)

    return ", ".join(cleaned)


def _extract_sentences(structure_json):
    """
    Lấy toàn bộ sentence.text thành list[str]
    """
    if not structure_json:
        return []
    
    structure_json = json.loads(structure_json)

    sentences = structure_json.get("sentences", [])

    return [
        s["text"].strip()
        for s in sentences
        if s.get("text")
    ]


def format_vbpl_dataset(dataset):
    """
    Parameters
    ----------
    dataset : datasets.Dataset

    Returns
    -------
    pandas.DataFrame
    """

    rows = []

    for sample in dataset:
        try:
            rows.append({
                "id": sample.get("item_id", np.nan),
                "scope": sample.get("scope"),
                "subject_title": sample.get("legal_area"),
                "issue_date": sample.get("issue_date"),
                "source_url": sample.get("source_url"),
                "api_url": sample.get("api_url"),
                "legal_type": sample.get('legal_type'),
                "legal_area": sample.get('legal_area'),
                "doc_number": _normalize_doc_number(
                    sample.get("doc_number")
                ),
                "title": sample.get("title"),
            })
        except:
            pass

    return pd.DataFrame(rows)

In [4]:
df = format_vbpl_dataset(ds)
df.head(5)

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title
0,1,trung_uong,Chưa phân loại,1950-03-27,https://vbpl.vn/van-ban/chi-tiet/nghi-dinh-so-...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị định,Chưa phân loại,24/LĐ-NĐ,Tổ chức các cơ quan Lao động địa phương liên k...
1,10,trung_uong,Chưa phân loại,1950-10-16,https://vbpl.vn/van-ban/chi-tiet/thong-tu-so-4...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Thông tư,Chưa phân loại,41-NV-6-TT,Định thể lệ xếp công chức vào thang lương chun...
2,100,trung_uong,Chưa phân loại,1950-05-14,https://vbpl.vn/van-ban/chi-tiet/sac-lenh-so-6...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Sắc lệnh,Chưa phân loại,68/SL,thành lập Ban Kinh tế Chính phủ
3,1000,trung_uong,Chưa phân loại,1957-01-22,https://vbpl.vn/van-ban/chi-tiet/nghi-quyet-so...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Nghị quyết,Chưa phân loại,Khôngsố,Về việc hoàn toàn tín nhiệm Chính phủ
4,10000,trung_uong,Chưa phân loại,1995-01-25,https://vbpl.vn/van-ban/chi-tiet/chi-thi-so-64...,https://vbpl-bientap-gateway.moj.gov.vn/api/qt...,Chỉ thị,Chưa phân loại,64-TTg,"Về tăng cường công tác giải quyết khiếu nại, t..."


In [8]:
df['legal_type'].value_counts()

legal_type
Quyết định                      84962
Nghị quyết                      26086
Thông tư                        16212
Bản dịch văn bản                10576
Chỉ thị                          7908
Nghị định                        4690
Thông tư liên tịch               3393
Văn bản hợp nhất                 1788
Sắc lệnh                          980
Công văn                          708
Luật                              580
Lệnh                              376
Pháp lệnh                         187
Văn bản hành chính liên quan      169
Văn bản liên quan                  43
Chương trình                       39
Nghị quyết liên tịch               33
Hiệp định                          17
Bộ luật                            16
Văn bản khác                        9
Thông báo                           8
Hiến pháp                           6
Thông tư liên bộ                    6
Sắc luật                            4
Chưa xác định                       4
Nghị định thư                       2
B

In [ ]:
from bs4 import BeautifulSoup

fail_indices = []

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/xml, text/xml, */*; q=0.01'
}


def extract_content_html(html_content:str) -> str :
    if not html_content :
        return None

    soup = BeautifulSoup(html_content, "html.parser")

    # Hàm đệ quy để duyệt và làm sạch DOM
    def clean_node(node):
        if node.name in ["style", "script", "link", "colgroup", "col"]:
            node.decompose()
            return

        # Nếu gặp bảng, ta giữ lại cấu trúc chính và xóa sạch mọi thuộc tính (style, border, width...)
        if node.name in ["table", "tbody", "tr", "td"]:
            node.attrs = {}  # Xóa sạch thuộc tính rác
            for child in list(node.children):
                clean_node(child)
            return

        # Nếu là các thẻ chứa văn bản bình thường (không nằm trong table)
        if node.name and node.name not in ["table", "tbody", "tr", "td"]:
            # Lấy toàn bộ text thô bên trong (đã tự động bỏ qua <b>, <i>, <span>, <br>...)
            text_content = node.get_text(strip=True)

            if text_content:
                # Tạo một thẻ <p> mới chứa text thô thuần túy
                new_p = soup.new_tag("p")
                new_p.string = text_content
                node.replace_with(new_p)
            else:
                # Nếu thẻ rỗng thì xóa bỏ
                node.decompose()


    # Chạy làm sạch từ thẻ body
    if soup.body:
        # Duyệt qua các con trực tiếp của body
        for child in list(soup.body.children):
            clean_node(child)

    # Loại bỏ các thẻ <p> trùng lặp nội dung hoặc rỗng phát sinh trong quá trình dọn dẹp
    final_elements = []
    seen_texts = set()

    for element in soup.body.children:
        if element.name == "p":
            txt = element.get_text(strip=True)
            if txt and txt not in seen_texts:
                final_elements.append(element)
                seen_texts.add(txt)
        elif element.name == "table":
            final_elements.append(element)

    # Gộp lại thành một body sạch sẽ
    new_body = soup.new_tag("body")
    for el in final_elements:
        new_body.append(el)

    return new_body.prettify()

def split_articles(clean_html_content: str):
    if not clean_html_content :
        return None

    # Khởi tạo BeautifulSoup để xử lý DOM văn bản thuần
    soup = BeautifulSoup(clean_html_content, "html.parser")

    # 1. Xóa thẻ <table> cuối cùng trong body (phần chữ ký) nếu có
    if soup.body:
        tables = soup.body.find_all("table")
        if tables:
            tables[-1].decompose()  # Xóa bỏ bảng cuối cùng

    # 2. Chuyển đổi thành văn bản thuần bằng cách gộp các thẻ <p> qua dấu xuống dòng
    # get_text(separator='\n') giúp mỗi block <p> cách nhau bằng một dấu xuống dòng rõ ràng
    raw_text = soup.get_text(separator="\n")

    # Làm sạch khoảng trắng thừa của toàn bộ văn bản
    lines = [line.strip() for line in raw_text.split("\n") if line.strip()]
    full_text = "\n".join(lines)

    # 3. Tiến hành tách Điều luật theo quy tắc i tăng dần liên tiếp (Điều 1, Điều 2, Điều 3...)
    articles_list = []
    current_expected_id = 1

    while True:
        # Tạo pattern tìm kiếm chính xác vị trí bắt đầu của "Điều x." hoặc "Điều x -"
        # \b nhằm đảm bảo không dính vào các từ như "Điều hành", "Điều khiển"
        current_pattern = rf"\bĐiều\s+{current_expected_id}\b"

        # Tìm vị trí của Điều hiện tại trong văn bản
        match_current = re.search(current_pattern, full_text, re.IGNORECASE)

        if not match_current:
            # Nếu không tìm thấy Điều tiếp theo (ví dụ tìm Điều 4 không thấy sau khi đã có Điều 3)
            # đồng nghĩa với việc ta đã duyệt hết toàn bộ các Điều luật tăng dần hợp lệ.
            break

        # Xác định điểm bắt đầu của nội dung Điều hiện tại
        start_pos = match_current.start()

        # Tìm kiếm Điều kế tiếp (i + 1) để làm điểm kết thúc cho Điều hiện tại
        next_pattern = rf"\bĐiều\s+{current_expected_id + 1}\b"
        match_next = re.search(next_pattern, full_text, re.IGNORECASE)

        if match_next:
            # Nếu có Điều tiếp theo, cắt văn bản từ đầu Điều này đến trước Điều sau
            end_pos = match_next.start()
            article_content = full_text[start_pos:end_pos].strip()
        else:
            # Nếu đây là Điều cuối cùng, lấy phần còn lại của văn bản
            article_content = full_text[start_pos:].strip()

            # Xử lý chặn lỗi: Nếu Điều cuối chứa ký tự dừng "./." thì cắt bỏ từ vị trí đó
            stop_marker = "./."
            last_article_content = article_content.split("./.")[0] if "./." in stop_marker else article_content

            # Xử lý bỏ dòng "Ai đó (Đã ký)" nếu có
            lines = last_article_content.split("\n")
            while lines and "(đã ký)" in lines[-1].lower():
                lines.pop()
            
            last_article_content = "\n".join(lines)

        # Lưu Điều luật đã bóc tách vào danh sách
        articles_list.append(article_content)

        # Tăng chỉ số mong đợi lên 1 để tìm Điều tiếp theo ở vòng lặp sau
        current_expected_id += 1

    return articles_list

def parse_document(soup: BeautifulSoup) -> dict:
    
    # Kiểm tra xem thẻ <data> có tồn tại không
    data_tag = soup.find("data")
    if not data_tag:
        return None

    # 1. Trích xuất các trường từ gốc <data>
    # Dùng find(recursive=False) để chỉ lấy thẻ con trực tiếp, tránh trùng với id của documentContent/documentIssues...
    doc_id = (
        data_tag.find("id", recursive=False).text
        if data_tag.find("id", recursive=False)
        else None
    )
    docs_code = (
        data_tag.find("docNum").text if data_tag.find("docNum") else None
    )
    article_title = (
        data_tag.find("title").text if data_tag.find("title") else None
    )
    issue_date = (
        data_tag.find("issueDate").text if data_tag.find("issueDate") else None
    )
    eff_from = (
        data_tag.find("effFrom").text if data_tag.find("effFrom") else None
    )
    eff_to = data_tag.find("effTo").text if data_tag.find("effTo") else None
    agency_name = (
        data_tag.find("agencyName").text if data_tag.find("agencyName") else None
    )

    # Lấy status từ effStatus -> name
    eff_status_tag = data_tag.find("effStatus")
    status = (
        eff_status_tag.find("name").text
        if eff_status_tag and eff_status_tag.find("name")
        else None
    )

    # 2. Trường documentContent -> content (content_text)
    doc_content_tag = data_tag.find("documentContent")
    content_text = (
        doc_content_tag.find("content").text
        if doc_content_tag and doc_content_tag.find("content")
        else None
    )
    if content_text == None :
        return None
    
    content_text = split_articles(extract_content_html(content_text))
    if not content_text :
        return None

    # 3. Trường documentMajors -> name (topic_title)
    doc_majors_tag = data_tag.find("documentMajors")
    topic_title = (
        doc_majors_tag.find("name").text
        if doc_majors_tag and doc_majors_tag.find("name")
        else None
    )

    # 4. Trường documentFields -> name (subject_title)
    doc_fields_tag = data_tag.find("documentFields")
    subject_title = (
        doc_fields_tag.find("name").text
        if doc_fields_tag and doc_fields_tag.find("name")
        else None
    )

    # 5. Trường references (Nằm trong thẻ <references>)
    # Trích xuất danh sách các tuple (id, docNum) từ targetDocument
    references_list = []
    references_tag = data_tag.find("references")
    if references_tag:
        # Tìm tất cả các thẻ <references> con bên trong
        for ref in references_tag.find_all("references"):
            target_doc = ref.find("targetDocument")
            if target_doc:
                ref_id = (
                    target_doc.find("id").text
                    if target_doc.find("id")
                    else None
                )
                ref_doc_num = (
                    target_doc.find("docNum").text
                    if target_doc.find("docNum")
                    else None
                )
                if ref_id or ref_doc_num:
                    references_list.append((ref_id, ref_doc_num))

    # Tổng hợp thành dictionary
    result_dict = {
        "id": doc_id,
        "docs_code": _normalize_doc_number(docs_code),
        "article_title": article_title,
        "issueDate": issue_date,
        "effFrom": eff_from,
        "effTo": eff_to,
        "status": status,
        "agency_name": agency_name,
        "topic_title": topic_title,
        "subject_title": subject_title,
        "references": references_list,  # Lưu dưới dạng list của các tuple
        "content_text": content_text,
    }

    return result_dict

def repair_work(row:pd.Series):
    item_id = str(row["id"])
    api_url = str(row['api_url'])

    # print(item_id)
    
    if not api_url or api_url == 'nan':
        if not item_id :
            fail_indices.append(row.name)
        else :
            api_url = f'https://vbpl-bientap-gateway.moj.gov.vn/api/qtdc/public/doc/{item_id}'

    try:
        # Timeout 10s để tránh treo vĩnh viễn
        res = requests.get(api_url, headers=HEADERS, timeout=30) 
        
        xml_soup = BeautifulSoup(res.content, 'xml')

        results = parse_document(xml_soup)
        if results is not None and results.get('status') == 'Hết hiệu lực toàn bộ' :
            return None
        return results
    except requests.exceptions.Timeout:
        fail_indices.append(row.name)
        # print(f"Timout at {row.name}")
        return None
    except Exception:
        fail_indices.append(row.name)
        # print(f"Error at {row.name}")
        return None



In [56]:
"(đã ký)" in "KT. BỘ TRƯỞNGTHỨ TRƯỞNG(Đã ký)Ngô Thế Dân".lower()

True

In [41]:
results = repair_work(df.iloc[75461])

if results is not None :
    for x in results['content_text'] :
        print(x)
        print("-"*10)
else :
    print(results)

None


In [42]:
import pandas as pd

repaired_records = []

# Gộp danh sách thành chuỗi Regex
regex_pattern = "|".join(['Luật', 'Bộ luật', 'Hiến pháp', 'Pháp lệnh', 'Nghị quyết', 'Nghị quyết liên tịch', 'Nghị định', 'Thông tư', 'Thông tư liên tịch', 'Thông tư liên bộ', 'Quyết định', 'Văn bản hợp nhất', 'Công văn', 'Hiệp định', 'Nghị định thư', 'Chưa xác định'])

# # Init danh sách task
df_left = df[
    (df['scope'] != 'dia_phuong')
    & (~df['id'].astype(str)
                .str.startswith("vbpqta"))
    & (
        df["legal_type"].fillna("Chưa xác định")
                    .str.contains(regex_pattern)
    )
    ]
len(df_left)

41439

In [43]:
from concurrent import futures as thread_futures
from tqdm.notebook import tqdm


MAX_WORKERS = 16

# Biến đếm để hiển thị trên thanh tqdm
success_count = 0
fail_count = 0
article_count = 0

for _ in range(3) :
    if len(df_left) == 0 : break

    fail_indices = []

    with thread_futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # 2. Truyền row (kiểu pd.Series) vào hàm. Lúc này row.name sẽ là chỉ số iloc
        futures = [
            executor.submit(repair_work, row) for _, row in df_left.iterrows()
        ]

        # 3. Tạo thanh tiến trình với cấu hình postfix ban đầu
        pbar = tqdm(
            thread_futures.as_completed(futures), total=len(futures), desc="Crawling"
        )

        for future in pbar:
            result = future.result()

            if result is not None:
                success_count += 1
                article_count += len(result['content_text'])
                repaired_records.append(result)
            else:
                fail_count += 1

            # 4. Cập nhật postfix theo thời gian thực
            pbar.set_postfix(OK=success_count, Articles=article_count, Fail=fail_count, Redo=len(fail_indices))

    # 5. Chuyển list dict thành DataFrame hoàn chỉnh
    final_df = pd.DataFrame(repaired_records)
    df_left = df.loc[fail_indices]

Crawling:   0%|          | 0/41439 [00:00<?, ?it/s]

Timout at 1092
Timout at 2167
Timout at 3214
Timout at 6783
Timout at 7141
Timout at 7193
Timout at 7492
Timout at 7592
Timout at 8375
Timout at 8784
Timout at 9338
Timout at 9934
Timout at 9976
Timout at 11622
Timout at 14763
Timout at 17339
Timout at 22408
Timout at 25415
Timout at 26469
Timout at 28567
Timout at 30121
Timout at 31078
Timout at 32947
Timout at 34376
Timout at 34921
Timout at 35673
Timout at 36147
Timout at 41320
Timout at 42907
Timout at 43153
Timout at 43794
Timout at 50246
Timout at 51550
Timout at 52402
Timout at 52826
Timout at 53219
Timout at 54308
Timout at 57778
Timout at 59498
Timout at 64163
Timout at 66367
Timout at 67613
Timout at 70786
Timout at 78997
Timout at 79046
Timout at 79183
Timout at 83044
Timout at 83853
Timout at 86098
Timout at 86992
Timout at 88534
Timout at 89517
Timout at 90067
Timout at 91508
Timout at 95295
Timout at 98438
Timout at 98621
Timout at 102342
Timout at 104288
Timout at 114127
Timout at 115021
Timout at 121986
Timout at 123713

Crawling:   0%|          | 0/74 [00:00<?, ?it/s]

In [44]:
df_left.head(5)

,id,scope,subject_title,issue_date,source_url,api_url,legal_type,legal_area,doc_number,title


In [47]:
final_df.sample(8)

,id,docs_code,article_title,issueDate,effFrom,effTo,status,agency_name,topic_title,subject_title,references,content_text
4681,146942,92/2017/NĐ-CP,"Nghị định số 92/2017/NĐ-CP Sửa đổi, bổ sung mộ...",2017-08-07T00:00:00,2017-09-25T00:00:00,2025,Hết hiệu lực một phần,Chính phủ,NaN,Chưa phân loại,"[(70800, 80/2015/QH13), (70819, 77/2015/QH13),...","[Điều 1. Sửa đổi, bổ sung một số điều, khoản c..."
14185,5503,788-TTg,Quyết định số 788-TTg Về việc người Việt Nam đ...,1997-09-24T00:00:00,1997-10-05T00:00:00,2002,Còn hiệu lực,Thủ tướng Chính phủ,NaN,Chưa phân loại,"[(11226, Không số)]",[Điều 1. Khuyến khích và tạo điều kiện để vận ...
11628,23645,1436/QĐ-TTg,Quyết định số 1436/QĐ-TTg Phê duyệt điều chỉnh...,2009-09-10T00:00:00,2009-09-10T00:00:00,2018,Còn hiệu lực,Thủ tướng Chính phủ,NaN,Chưa phân loại,"[(18126, 35/2005/QH11), (22843, 32/2001/QH10)]",[Điều 1.Phê duyệt điều chỉnh Quy hoạch tổng th...
14057,47240,53/2014/TT-BCT,Thông tư số 53/2014/TT-BCT Quy định điều kiện ...,2014-12-18T00:00:00,2015-02-03T00:00:00,2017,Hết hiệu lực một phần,Bộ Công Thương,Công Thương,Khoa học công nghệ,"[(25495, 55/2010/QH12), (27878, 95/2012/NĐ-CP)...",[Điều 1. Phạm vi điều chỉnh\nThông tư này quy ...
243,104434,30/2014/TT-BNNPTNT,Thông tư số 30/2014/TT-BNNPTNT Ban hành Danh m...,2014-09-05T00:00:00,2015-01-01T00:00:00,2015,Còn hiệu lực,Bộ Nông nghiệp và Phát triển nông thôn,Nông nghiệp và phát triển nông thôn,NaN,"[(32909, 41/2013/QH13), (27765, 39/2012/TT-BNN...",[Điều 1. Danh mục vật thể thuộc diện kiểm dịch...
14056,47324,560/QĐ-BNN-CB,Quyết định số 560/QĐ-BNN-CB Về việc ban hành q...,2011-03-24T00:00:00,2011-03-24T00:00:00,2014,Còn hiệu lực,Bộ Nông nghiệp và Phát triển nông thôn,Nông nghiệp và phát triển nông thôn,Lâm nghiệp,"[(25185, 04/2004/TT-BCA), (25837, 109/2010/NĐ-...",[Điều 1.Ban hành kèm theo Quyết định này Quy đ...
1470,113352,103/2007/NĐ-CP,Nghị định số 103/2007/NĐ-CP Quy định xử lý trá...,2007-06-14T00:00:00,2007-07-23T00:00:00,2010,Hết hiệu lực một phần,Chính phủ,Nội vụ,"Công chức, viên chức","[(21216, 11/2003/PL-UBTVQH11), (5778, 21/2000/...",[Điều 1.Phạm vi điều chỉnh\nNghị định này quy ...
5491,153633,01/2022/TT-BYT,Thông tư số 01/2022/TT-BYT Quy định ghi chép b...,2022-01-10T00:00:00,2022-03-01T00:00:00,2026,Hết hiệu lực một phần,Bộ Y tế,NaN,Chưa phân loại,"[(20981, 06/2003/PL-UBTVQH11), (113074, 97/201...",[Điều 1. Phạm vi điều chỉnh và đối tượng áp dụ...


In [54]:
final_df.to_parquet("data/DONE_CRAWL.parquet")
final_df.sample(5).to_json("data/SAMPLE_CRAWL.json", orient="records", force_ascii=False, indent=4)
if len(df_left) > 0 :
    df_left.to_parquet("data/REDO_CRAWL.parquet")